In [1]:
from plotly import graph_objects as go
from freqtrade.plot.plotting import  generate_candlestick_graph
from scipy.signal import argrelextrema
import pandas as pd
import numpy as np
import os
from pathlib import Path
from freqtrade.data.history import load_pair_history
from freqtrade.enums import CandleType
from freqtrade.configuration import Configuration
from sklearn.cluster import HDBSCAN
from sklearn import preprocessing
from sklearn.linear_model import LinearRegression
import random
import datetime
from sklearn.metrics import silhouette_score
import json
from sklearn.cluster import KMeans
from kneed import DataGenerator, KneeLocator

In [2]:
project_root = "somedir/freqtrade"
i=0
try:
    os.chdirdir(project_root)
    assert Path('docker-compose.yml').is_file()
except:
    while i<4 and (not Path('docker-compose.yml').is_file()):
        os.chdir(Path(Path.cwd(), '../'))
        i+=1
    project_root = Path.cwd()
print(Path.cwd())

/root/trade


In [3]:
# config = Configuration.from_files([])
config = Configuration.from_files(["user_data/break_config.json"])

config["timeframe"] = "15m"
config["strategy"] = "ReactionBreakStrategy"
data_location = config["datadir"]
pair = "DYDX/USDT:USDT"

def random_color_generator():
    r = random.randint(0, 255)
    g = random.randint(0, 255)
    b = random.randint(0, 255)
    return f"rgb({r}, {g}, {b})"

In [ ]:
!freqtrade list-pairs --print-json > kucoin_btc_pairs.json --quote BTC --trading-mode spot --exchange kucoin

In [ ]:
!freqtrade download-data -t 15m -c user_data/break_config.json

In [84]:
with open("selected_pairs.json") as file:
    pairs = json.load(file)

In [5]:
new_pairs = dict(pair=[], coef=[], intercept=[])
for pair in pairs:
    df = load_pair_history(
        datadir=data_location,
        timeframe=config["timeframe"],
        pair=pair,
        data_format = "feather",
        candle_type=CandleType.SPOT,
    )
    X = df.index.values.reshape(-1,1)
    y = df.close.values
    model = LinearRegression()
    model.fit(X, y)
    new_pairs['pair'].append(pair)
    new_pairs['coef'].append(model.coef_[0])
    new_pairs['intercept'].append(model.intercept_)
df_ = pd.DataFrame(new_pairs)

# print(f"Loaded {len(candles)} rows of data for {pairs[0]} from {data_location}")

In [6]:
pairs = [pair.replace('/BTC','/USDT') for pair in df_[df_.coef > 0].sort_values(by=['intercept'])['pair'].values]
pairs_str = ' '.join(pairs)

In [132]:
!freqtrade download-data -t 5m -p CKB/USDT --exchange kucoin --trading-mode spot

2024-03-06 15:34:56,178 - freqtrade - INFO - freqtrade 2024.1
2024-03-06 15:34:56,178 - freqtrade.loggers - INFO - Verbosity set to 0
2024-03-06 15:34:56,178 - freqtrade.configuration.configuration - INFO - Using exchange kucoin
2024-03-06 15:34:56,179 - freqtrade.configuration.configuration - INFO - Using user-data directory: /root/trade/user_data ...
2024-03-06 15:34:56,179 - freqtrade.configuration.configuration - INFO - Using data directory: /root/trade/user_data/data/kucoin ...
2024-03-06 15:34:56,179 - freqtrade.configuration.configuration - INFO - Using pairs ['CKB/USDT']
2024-03-06 15:34:56,179 - freqtrade.configuration.configuration - INFO - timeframes --timeframes: ['5m']
2024-03-06 15:34:56,179 - freqtrade.configuration.configuration - INFO - Detected --trading-mode: spot
2024-03-06 15:34:56,180 - freqtrade.exchange.check_exchange - INFO - Checking exchange...
2024-03-06 15:34:56,184 - freqtrade.exchange.check_exchange - WARNING - Exchange "kucoin" is known to the the ccxt l

In [85]:
targets = []
i = 0

In [88]:
pair = pairs[i]
i += 1
print(f"Generating plot for pair {pair}")
df = load_pair_history(
    datadir=data_location,
    timeframe=config["timeframe"],
    pair=pair,
    data_format = "feather",
    candle_type=CandleType.SPOT,
)
X = df.index.values.reshape(-1,1)
y = df.close.values
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)
fig = generate_candlestick_graph(pair=pair, data=df)
fig.add_trace(go.Scatter(x=df['date'],y=y_pred, mode = 'lines', marker_color='black'))
fig.show()

Generating plot for pair HEART/USDT


In [80]:
targets.append(pair)

In [112]:
targets

[]

In [83]:
with open("selected_pairs.json", "w") as outfile: 
    json.dump(targets, outfile)

In [113]:
with open("selected_pairs.json") as file:
    pairs = json.load(file)

In [114]:
pairs

['RSR/USDT',
 'CKB/USDT',
 'HEART/USDT',
 'ONE/USDT',
 'ZIL/USDT',
 'LMR/USDT',
 'HAI/USDT',
 'NWC/USDT',
 'NOIA/USDT',
 'ALGO/USDT',
 'PUSH/USDT',
 'ENJ/USDT',
 'SCRT/USDT',
 'FTM/USDT']

In [139]:
pair = pairs[0]

In [87]:
# window = 10
# df = df.set_index('date', drop=False)
# df = df[df.date >= df[df.labels==df.labels.iat[-1]].date.iat[0]]
# start = datetime.date.today() - datetime.timedelta(days=days)
# df = df[start.strftime('%Y-%m-%d'):datetime.date.today().strftime('%Y-%m-%d')]
# df = df['2024-02-15':'2024-03-03']
# df['min_values'] = df.iloc[argrelextrema(df.low.values, np.less_equal,order=window)[0]]['low']
# df['max_values'] = df.iloc[argrelextrema(df.high.values, np.greater_equal,order=window)[0]]['high']
# df['max_volume'] = df.iloc[argrelextrema(df.volume.values, np.greater_equal,order=window)[0]]['close']
# df['extrema'] = df['max_values'].fillna(df['min_values'])
# df['extrema'] = df['extrema'].fillna(df['max_volume'])
X = df['close'].values.reshape(-1,1)
# scaler = preprocessing.StandardScaler().fit(X_train)
# X_scaled = scaler.transform(X_train)
# hdb = HDBSCAN(min_cluster_size=302)
# hdb.fit(X)
# df['label'] = hdb.labels_
# df['probability'] = hdb.probabilities_

In [107]:
df = df[df.cluster==df.cluster.iat[-1]]

In [170]:
!freqtrade download-data -t 15m -p RSR/USDT --exchange kucoin --trading-mode spot

2024-03-06 16:36:09,125 - freqtrade - INFO - freqtrade 2024.1
2024-03-06 16:36:09,126 - freqtrade.loggers - INFO - Verbosity set to 0
2024-03-06 16:36:09,126 - freqtrade.configuration.configuration - INFO - Using exchange kucoin
2024-03-06 16:36:09,127 - freqtrade.configuration.configuration - INFO - Using user-data directory: /root/trade/user_data ...
2024-03-06 16:36:09,127 - freqtrade.configuration.configuration - INFO - Using data directory: /root/trade/user_data/data/kucoin ...
2024-03-06 16:36:09,127 - freqtrade.configuration.configuration - INFO - Using pairs ['RSR/USDT']
2024-03-06 16:36:09,127 - freqtrade.configuration.configuration - INFO - timeframes --timeframes: ['15m']
2024-03-06 16:36:09,127 - freqtrade.configuration.configuration - INFO - Detected --trading-mode: spot
2024-03-06 16:36:09,128 - freqtrade.exchange.check_exchange - INFO - Checking exchange...
2024-03-06 16:36:09,133 - freqtrade.exchange.check_exchange - WARNING - Exchange "kucoin" is known to the the ccxt 

In [171]:
df = load_pair_history(
    datadir=data_location,
    timeframe='15m',
    pair=pair,
    data_format = "feather",
    candle_type=CandleType.SPOT,
)
df = df[df.date>'2024-03-05']

In [173]:
X = df['close'].values.reshape(-1,1)
sum_of_squared_distances = []
K = range(1,15)
for k in K:
    km = KMeans(n_clusters=k)
    km = km.fit(X)
    sum_of_squared_distances.append(km.inertia_)
kn = KneeLocator(K, sum_of_squared_distances,S=1.0, curve="convex", direction="decreasing")
kmeans = KMeans(n_clusters=kn.knee).fit(X)
df['cluster'] = kmeans.predict(X)
fig = generate_candlestick_graph(pair=pair, data=df)
# fig = go.Figure()
# fig.add_trace(go.Scatter(x=df['date'],y=df['close'], mode = 'markers', marker_color='black'))
# fig.add_trace(go.Scatter(x=df['date'],y=df['max_volume'], mode = 'markers', marker_color='black'))
for cluster in df.cluster.unique():
    color = random_color_generator()
    df_1 = df[df.cluster == cluster]
    Max = df_1.close.max()
    Min = df_1.close.min()
    fig.add_hline(y=Max, line_width=1, line_color=color)
    fig.add_hline(y=Min, line_width=1, line_color=color)
    fig.add_hline(y=0.005671, line_width=1, line_color='green')
    fig.add_hline(y=0.005584, line_width=1, line_color='red')
# fig.add_hline(y=Max, line_width=1, line_color='red')
# fig.add_hline(y=Min, line_width=1, line_color='red')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [159]:
high = 0.005428
low = 0.005135
(high - low)/low * 100

5.70593962999026

In [166]:
100 / 0.005428

18422.99189388357

In [191]:
kmeans.score(X)

-0.8750920877118233

In [161]:
df.cluster.iat[0]

0

In [67]:
df = df[df.label==df.label.iat[-1]]
X = df['close'].values.reshape(-1,1)
hdb = HDBSCAN(min_cluster_size=260)
hdb.fit(X)
df['label'] = hdb.labels_
df['probability'] = hdb.probabilities_

2

In [65]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
import hdbscan
param_dist = {'min_samples': [10,30,50,60,100],
              'min_cluster_size':[100,200,300,400,500,600],  
              'cluster_selection_method' : ['eom','leaf'],
              'metric' : ['euclidean','manhattan'] 
             }

In [ ]:
hdb = hdbscan.HDBSCAN(gen_min_span_tree=True).fit(X)
validity_scorer = make_scorer(hdbscan.validity.validity_index,greater_is_better=True)

n_iter_search = 20
random_search = RandomizedSearchCV(hdb
                                   ,param_distributions=param_dist
                                   ,n_iter=n_iter_search
                                   ,scoring=validity_scorer 
                                   ,random_state=42)

random_search.fit(X)

print(f"Best Parameters {random_search.best_params_}")
print(f"DBCV score :{random_search.best_estimator_.relative_validity_}")

In [ ]:
from freqtrade.resolvers import StrategyResolver
from freqtrade.data.dataprovider import DataProvider
strategy = StrategyResolver.load_strategy(config)
strategy.dp = DataProvider(config, None, None)
strategy.ft_bot_start()
df = strategy.analyze_ticker(candles, {'pair': pair})
data = df.set_index('date', drop=False)
print(f"Generated {df['enter_long'].sum()} entry signals")

In [ ]:
from freqtrade.data.btanalysis import load_backtest_data, load_backtest_stats

# if backtest_dir points to a directory, it'll automatically load the last backtest file.
backtest_dir = config["user_data_dir"] / "backtest_results"
# backtest_dir can also point to a specific file 
# backtest_dir = config["user_data_dir"] / "backtest_results/backtest-result-2020-07-01_20-04-22.json"

## Backtest

In [ ]:
# You can get the full backtest statistics by using the following command.
# This contains all information used to generate the backtest result.
stats = load_backtest_stats(backtest_dir)

strategy = 'SampleStrategy'
# All statistics are available per strategy, so if `--strategy-list` was used during backtest, this will be reflected here as well.
# Example usages:
print(stats['strategy'][strategy]['results_per_pair'])
# Get pairlist used for this backtest
print(stats['strategy'][strategy]['pairlist'])
# Get market change (average change of all pairs from start to end of the backtest period)
print(stats['strategy'][strategy]['market_change'])
# Maximum drawdown ()
print(stats['strategy'][strategy]['max_drawdown'])
# Maximum drawdown start and end
print(stats['strategy'][strategy]['drawdown_start'])
print(stats['strategy'][strategy]['drawdown_end'])


# Get strategy comparison (only relevant if multiple strategies were compared)
print(stats['strategy_comparison'])


In [ ]:
# Load backtested trades as dataframe
trades = load_backtest_data(backtest_dir)

# Show value-counts per pair
trades.groupby("pair")["exit_reason"].value_counts()

## Plotting daily profit / equity line

In [ ]:
# Plotting equity line (starting with 0 on day 1 and adding daily profit for each backtested day)

from freqtrade.configuration import Configuration
from freqtrade.data.btanalysis import load_backtest_stats
import plotly.express as px
import pandas as pd

# strategy = 'SampleStrategy'
# config = Configuration.from_files(["user_data/config.json"])
# backtest_dir = config["user_data_dir"] / "backtest_results"

stats = load_backtest_stats(backtest_dir)
strategy_stats = stats['strategy'][strategy]

df = pd.DataFrame(columns=['dates','equity'], data=strategy_stats['daily_profit'])
df['equity_daily'] = df['equity'].cumsum()

fig = px.line(df, x="dates", y="equity_daily")
fig.show()


### Load live trading results into a pandas dataframe

In case you did already some trading and want to analyze your performance

In [4]:
from freqtrade.data.btanalysis import load_trades_from_db

# Fetch trades from database
trades = load_trades_from_db("sqlite:///user_data/ReactionBreakStrategy_dry.sqlite")

# Display results
trades.groupby("pair")["exit_reason"].value_counts()

pair            exit_reason       
CHR/USDT:USDT   trailing_stop_loss    15
                stop_loss              1
DUSK/USDT:USDT  trailing_stop_loss     1
MINA/USDT:USDT  trailing_stop_loss    19
                stop_loss              1
RNDR/USDT:USDT  trailing_stop_loss    19
                stop_loss              1
Name: count, dtype: int64

In [5]:
# trades_red = trades.loc[trades['pair'] == pair]

## Analyze the loaded trades for trade parallelism
This can be useful to find the best `max_open_trades` parameter, when used with backtesting in conjunction with `--disable-max-market-positions`.

`analyze_trade_parallelism()` returns a timeseries dataframe with an "open_trades" column, specifying the number of open trades for each candle.

In [ ]:
from freqtrade.data.btanalysis import analyze_trade_parallelism

# Analyze the above
parallel_trades = analyze_trade_parallelism(trades, '5m')

parallel_trades.plot()

## Plot results

Freqtrade offers interactive plotting capabilities based on plotly.

In [ ]:
from freqtrade.plot.plotting import  generate_candlestick_graph
# Limit graph period to keep plotly quick and reactive

# Filter trades to one pair
trades_red = trades.loc[trades['pair'] == pair]

data_red = data['2019-06-01':'2019-06-10']
# Generate candlestick graph
graph = generate_candlestick_graph(pair=pair,
                                   data=data_red,
                                   trades=trades_red,
                                   indicators1=['sma20', 'ema50', 'ema55'],
                                   indicators2=['rsi', 'macd', 'macdsignal', 'macdhist']
                                  )
graph.show()

In [ ]:
# Show graph inline
graph.show()

# Render graph in a separate window
# graph.show(renderer="browser")


## Plot average profit per trade as distribution graph

In [ ]:
import plotly.figure_factory as ff

hist_data = [trades.profit_ratio]
group_labels = ['profit_ratio']  # name of the dataset

fig = ff.create_distplot(hist_data, group_labels, bin_size=0.01)
fig.show()


In [6]:
from freqtrade.data.btanalysis import load_trades_from_db
trades = load_trades_from_db("sqlite:///user_data/ReactionBreakStrategy_dry.sqlite")
trades.profit_ratio.sum()

-0.027612510795092537

In [ ]:
from scipy.signal import argrelextrema, find_peaks
import numpy as np
import pandas as pd
from plotly import graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
peaks = find_peaks(candles.close)
candles['min'] = candles.iloc[argrelextrema(candles.close.values, np.less_equal,order=5)[0]]['close']
candles['peaks'] = candles.iloc[find_peaks(candles.close,prominence=10)[0]]['close']

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=candles['date'],y=candles['close']))
fig.add_trace(go.Scatter(x=candles['date'],y=candles['peaks'],mode = 'markers'))
fig.add_hline(y=1.2719, line_width=1, line_dash="dash", line_color="green")
fig.add_hline(y=1.2308, line_width=1, line_dash="dash", line_color="green")
fig.show()

# fig = make_subplots(rows=2, cols=1)#, shared_xaxes=True)#, row_width=[0.2, 0.8])
# fig.add_trace(go.Scatter(x=candles['date'], y=candles['close'], name='Close price'), row=1, col=1)
# fig.add_trace(go.Scatter(x=candles['date'], y=candles['extrema'], name='Extrema', mode = 'markers'), row=1, col=1)
# fig.update_layout(autosize=True, height=700, xaxis=dict(rangeslider=dict(visible=False)))
# fig.show()

In [ ]:
import numpy as np
from sklearn.cluster import HDBSCAN

In [49]:
low_limit, high_limit = -1e-3, +1e-3
bins_width = 0.001
window = 8
threshold = 3
df = candles.copy()
df = df.set_index('date', drop=False)
df = df['2024-02-20':'2024-02-24']
df['min_values'] = df.iloc[argrelextrema(df.low.values, np.less_equal,order=window)[0]]['low'] + low_limit
df['max_values'] = df.iloc[argrelextrema(df.high.values, np.greater_equal,order=window)[0]]['high'] + high_limit
df['extrema'] = df['max_values'].fillna(df['min_values'])
df['new'] = 1
X = np.array(df[['new','extrema']].dropna().values.tolist())

In [ ]:
hdb = HDBSCAN()
hdb.fit(X)

In [90]:
df_ = pd.DataFrame({'values':df['extrema'].dropna().values ,'labels':hdb.labels_, 'probabilities':hdb.probabilities_})
fig = go.Figure()
fig.add_trace(go.Scatter(x=candles['date'],y=candles['close']))
for lable in [1,0]:
    Max = df_[(df_.labels == lable)]['values'].max()
    Min = df_[(df_.labels == lable)]['values'].min()
    fig.add_hline(y=Max, line_width=1, line_dash="dash")
    fig.add_hline(y=Min, line_width=1, line_dash="dash")
fig.show()